# 🚀 LLM 10분만에 파인튜닝하기 (Colab용)

이 노트북은 `TinyLlama-1.1B` 모델을 `LoRA`를 사용하여 빠르게 학습시키는 예제입니다.
**[런타임] > [런타임 유형 변경]에서 'T4 GPU'를 꼭 선택해주세요!**

In [ ]:
# 1. 필수 라이브러리 설치 (Colab에서는 이거 먼저 실행하세요)
!pip install -q -U torch transformers peft trl bitsandbytes datasets accelerate

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# 2. 모델 설정
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
new_model_name = "TinyLlama-Kor-Tuned"

print(f"Loading model: {model_name}...")

# 4비트 양자화 설정 (GPU 메모리 절약)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# 모델 로드
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
# 3. 데이터셋 준비
# 실습용 데이터셋 로드 (여기서는 guanaco 100개만 사용)
dataset = load_dataset("mlabonne/guanaco-llama2-1k", split="train[:100]")
print("Dataset loaded! Example:")
print(dataset[0]['text'][:100])

In [ ]:
# 4. LoRA 및 학습 파라미터 설정
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
)

training_params = SFTConfig(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    logging_steps=10,
    learning_rate=2e-4,
    max_seq_length=512,
    fp16=True, # T4 GPU 사용시 필수
    group_by_length=True,
    packing=False,
    report_to="none"
)

In [ ]:
# 5. 학습 시작!
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=training_params,
)

print("Starting training...")
trainer.train()

In [ ]:
# 6. 저장 및 확인
trainer.model.save_pretrained(new_model_name)
tokenizer.save_pretrained(new_model_name)
print(f"Training Done! Model saved to {new_model_name}")